In [ ]:
from pathlib import Path

DATA_DIR = Path("Data")

In [ ]:
import time
from googleapiclient.discovery import build
import googleapiclient.discovery
import pandas as pd
api_key = '' #A YouTube data API key is required for executing this code
youtube = build('youtube', 'v3', developerKey=api_key)
api_service_name = "youtube"
api_version = "v3"

**Detecting the language**

In [ ]:
from pathlib import Path
import csv

import torch
from googleapiclient.discovery import build
from transformers import pipeline


youtube = build(
    "youtube",
    "v3",
    developerKey=api_key
)


pipeline_device = 0 if torch.cuda.is_available() else -1

language_detector = pipeline(
    task="text-classification",
    model="papluca/xlm-roberta-base-language-detection",
    truncation=True,
    device=pipeline_device
)


def classify_language(text):
    """Detect the language of a text."""
    if text is None or not str(text).strip():
        return "unclassifiable"

    result = language_detector(str(text))[0]
    return result["label"]


base = Path("Data") / "Videos"

for csv_path in base.rglob("*.csv"):
    with csv_path.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.reader(f))

    if not rows:
        continue

    header = rows[0]
    data = rows[1:]

    video_ids = list(
        dict.fromkeys(
            row[0].strip()
            for row in data
            if row and row[0].strip()
        )
    )

    video_info = {}

    # The YouTube API accepts up to 50 video IDs per request.
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i + 50]

        try:
            response = youtube.videos().list(
                part="snippet,liveStreamingDetails",
                id=",".join(batch)
            ).execute()
        except Exception as exc:
            print(f"API error in {csv_path.name}, batch {i // 50 + 1}: {exc}")
            continue

        for item in response.get("items", []):
            snippet = item.get("snippet", {})
            live_details = item.get("liveStreamingDetails", {})

            title = snippet.get("title", "")
            description = snippet.get("description", "")

            combined_text = f"{title}. {description}".strip()
            detected_language = classify_language(combined_text)

            actual_start_time = live_details.get("actualStartTime", "")
            actual_end_time = live_details.get("actualEndTime", "")

            # Archived live stream: the broadcast started and has already ended.
            is_archived_live = bool(actual_start_time and actual_end_time)

            video_info[item["id"]] = {
                "video_title": title,
                "video_description": description,
                "detected_language": detected_language,
                "is_live_broadcast": bool(live_details),
                "is_archived_live": is_archived_live,
                "live_actual_start_time": actual_start_time,
                "live_actual_end_time": actual_end_time,
            }

    columns_to_add = [
        "video_title",
        "video_description",
        "detected_language",
        "is_live_broadcast",
        "is_archived_live",
        "live_actual_start_time",
        "live_actual_end_time",
    ]

    # Avoid duplicating columns if the script is run more than once.
    existing_indices = {
        column: header.index(column)
        for column in columns_to_add
        if column in header
    }

    new_columns = [
        column
        for column in columns_to_add
        if column not in header
    ]

    new_header = header + new_columns
    new_rows = []

    for row in data:
        if not row:
            continue

        # Ensure the row is long enough to update existing columns safely.
        row = row + [""] * (len(header) - len(row))

        video_id = row[0].strip()
        info = video_info.get(video_id, {})

        values = {
            "video_title": info.get("video_title", ""),
            "video_description": info.get("video_description", ""),
            "detected_language": info.get("detected_language", ""),
            "is_live_broadcast": info.get("is_live_broadcast", False),
            "is_archived_live": info.get("is_archived_live", False),
            "live_actual_start_time": info.get("live_actual_start_time", ""),
            "live_actual_end_time": info.get("live_actual_end_time", ""),
        }

        # Update columns that already exist.
        for column, index in existing_indices.items():
            row[index] = values[column]

        # Append columns that do not yet exist.
        row.extend(values[column] for column in new_columns)
        new_rows.append(row)

    with csv_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(new_header)
        writer.writerows(new_rows)

    archived_count = sum(
        bool(info.get("is_archived_live"))
        for info in video_info.values()
    )

    print(
        f"Updated: {csv_path} | "
        f"Videos retrieved: {len(video_info)}/{len(video_ids)} | "
        f"Archived live streams: {archived_count}"
    )

In [ ]:
import csv
from pathlib import Path

base = Path("Data/Videos")

sample_dirs = {
    "Scientific": base / "terms_cient",
    "Pseudoscientific": base / "terms_pseudo",
}

TRUE_VALUES = {"true", "1", "yes"}

for sample_name, sample_dir in sample_dirs.items():
    videos = {}

    for csv_path in sample_dir.rglob("*.csv"):
        with csv_path.open("r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)

            if not reader.fieldnames:
                continue

            first_column = reader.fieldnames[0]

            for row in reader:
                video_id = row.get("video_id", "").strip()

                if not video_id:
                    video_id = row.get(first_column, "").strip()

                if not video_id:
                    continue

                is_archived_live = (
                    row.get("is_archived_live", "")
                    .strip()
                    .lower()
                    in TRUE_VALUES
                )

                # Count each video only once within the sample.
                # If duplicated, preserve True if any copy identifies it as live.
                videos[video_id] = (
                    videos.get(video_id, False) or is_archived_live
                )

    total_videos = len(videos)
    archived_lives = sum(videos.values())

    percentage = (
        100 * archived_lives / total_videos
        if total_videos > 0
        else 0
    )

    print(
        f"{sample_name}: "
        f"{archived_lives}/{total_videos} archived live streams "
        f"({percentage:.1f}%)"
    )